# Session 06: 서비스 B - Gemini API로 리뷰 분석

## 학습 목표

서비스 A는 **학습된 ML 모델**로 예측하는 서비스였습니다.
서비스 B는 **LLM(Large Language Model)**을 호출하여 고객 리뷰를 분석하는 서비스입니다.

### ML 서비스 vs LLM 서비스 비교

| 비교 항목 | 서비스 A (ML 모델) | 서비스 B (LLM) |
|-----------|-------------------|----------------|
| 모델 위치 | 로컬 서버에 파일로 존재 | 외부 API (Google Cloud) |
| 추론 방식 | `pipeline.predict()` | HTTP API 호출 |
| 입력 형식 | 구조화된 숫자/범주형 데이터 | 자연어 텍스트 |
| 출력 형식 | 확률값 (결정적) | 자연어 텍스트 (비결정적) |
| 비용 구조 | 서버 리소스(CPU/RAM) | API 호출 당 과금 (토큰 기반) |
| 지연 시간 | 수 ms | 수백 ms ~ 수 s |

### 완성할 것들
- Gemini API 기본 사용법 익히기
- 프롬프트 엔지니어링으로 구조화된 JSON 응답 받기
- `schemas.py`, `gemini_client.py`, `main.py` 작성
- 서비스 B 테스트

In [ ]:
!pip install -q google-generativeai

In [ ]:
import os
import json
import google.generativeai as genai

os.environ["GEMINI_API_KEY"] = "your-api-key-here"  # ← 실습 시 본인 키로 교체
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

## 1. Gemini API 기본 사용법

### Google Generative AI 라이브러리

```python
import google.generativeai as genai

genai.configure(api_key="YOUR_KEY")
model = genai.GenerativeModel("gemini-2.5-flash")
response = model.generate_content("안녕하세요")
print(response.text)
```

### WHY Gemini 2.5 Flash?
- **빠른 응답**: Flash 모델은 Pro 대비 응답 속도가 빠름
- **저렴한 비용**: API 서비스에서는 호출당 비용이 중요
- **충분한 성능**: 리뷰 분석 같은 태스크에는 Flash로 충분

In [ ]:
# 여기에 코드를 작성하세요


## 2. 프롬프트 엔지니어링 - 구조화된 JSON 응답 받기

### WHY 프롬프트 엔지니어링이 중요한가?

LLM의 출력은 기본적으로 **자유 형식 텍스트**입니다. API 서비스에서는 이것을 **구조화된 JSON**으로 받아야 합니다.

```
"이 리뷰는 긍정적이네요. 배송에 대한 내용이고 만족도가 높습니다."
{"sentiment": "긍정", "category": "배송", "summary": "빠른 배송에 만족", "confidence": 0.95}
```

### 프롬프트 설계 원칙
1. **역할 지정**: "당신은 고객 리뷰 분석 전문가입니다"
2. **출력 형식 명시**: "반드시 JSON 형식으로 응답하세요"
3. **필드 설명**: 각 필드의 의미와 가능한 값을 명확히
4. **예시 제공**: 입력-출력 예시를 1~2개 포함

In [ ]:
# 여기에 코드를 작성하세요


## 3. schemas.py 작성

서비스 B의 스키마는 서비스 A와 구조는 비슷하지만, **LLM 서비스 특유의 고려사항**이 있습니다:

| 서비스 A (ML) | 서비스 B (LLM) |
|--------------|----------------|
| 13개 숫자/범주형 입력 | 1개 텍스트 입력 |
| 입력 범위 검증 (ge/le) | 텍스트 길이 검증 (min/max_length) |
| 결정적 출력 | 비결정적 출력 → confidence 필드로 불확실성 표현 |

In [ ]:
# 여기에 코드를 작성하세요


## 4. gemini_client.py 작성

### WHY 클라이언트를 별도 클래스로 분리하는가?

서비스 A의 `model.py`처럼, LLM 호출 로직을 별도 파일로 분리합니다.

| 분리하지 않으면 | 분리하면 |
|----------------|----------|
| main.py가 비대해짐 | 각 파일의 역할이 명확 |
| 프롬프트 수정 시 API 코드도 건드림 | 프롬프트만 독립적으로 수정 가능 |
| 테스트하기 어려움 | 클라이언트만 단독 테스트 가능 |

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


## 5. 샘플 리뷰 여러 건 테스트

다양한 감성(긍정/부정/중립)과 카테고리(품질/배송/가격/서비스)의 리뷰를 테스트하여 모델의 분석 능력을 확인합니다.

In [ ]:
# 여기에 코드를 작성하세요


## 정리

### 이번 세션에서 만든 것

| 파일 | 역할 |
|------|------|
| `schemas.py` | ReviewRequest/ReviewResponse 스키마 정의 |
| `gemini_client.py` | Gemini API 호출 + JSON 파싱 로직 |
| `main.py` | FastAPI 앱 + /health, /analyze 엔드포인트 |

### ML 서비스 vs LLM 서비스 - 배운 차이점

| 항목 | 서비스 A | 서비스 B |
|------|---------|----------|
| 모델 로딩 | 로컬 파일 (joblib) | 외부 API (HTTP) |
| 입력 검증 | 숫자 범위 (ge/le) | 텍스트 길이 (min/max_length) |
| 추론 | `predict_proba()` | 프롬프트 → API 호출 |
| 출력 파싱 | 불필요 (숫자) | JSON 파싱 필요 (비정형) |
| 에러 처리 | ValueError | JSONDecodeError + 폴백 |
